# Exp 8b: diagnostic on the Exp 8 checkpoints

Replays the unconstrained run from the saved checkpoints (and checks it matches them exactly), records data-gradient vs weight-decay terms during the replay, and reverts selected weight changes to see which ones carry the forgetting.

Needs the Exp 8 output attached as input (the folder or the zip). Exploratory and descriptive only.

CPU, 10-20 min.

In [ ]:
# 0. Install (Colab and Kaggle already have PyTorch)
!pip install -q torch_geometric

## 1. Locate the Experiment 8 results and load their configuration

In [ ]:
import os, json, time, random, shutil, datetime, itertools, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.utils import subgraph
from torch_geometric.nn import GCNConv
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

SMOKE_TEST = False
EXP8_DIR = None            # set a path here if automatic search fails, e.g. '/kaggle/input/exp8/exp8_confinement_curve'
INPUT_ROOTS = [r for r in ['/kaggle/input', '/kaggle/working', '/content', '.'] if os.path.isdir(r)]
import zipfile

def find_exp8():
    hits = []
    for root in INPUT_ROOTS:
        for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
            if 'exp8b' in dirpath: continue
            if 'config_fingerprint.json' in filenames and os.path.isdir(os.path.join(dirpath, 'checkpoints')):
                hits.append(dirpath)
    return sorted(set(hits))

def unzip_exp8():
    # if only the Experiment 8 ZIP was attached, extract it into the working folder
    for root in INPUT_ROOTS:
        for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
            for fn in filenames:
                if not fn.endswith('.zip') or 'exp8b' in fn: continue
                p = os.path.join(dirpath, fn)
                try:
                    with zipfile.ZipFile(p) as z:
                        names = z.namelist()
                        if any(n.endswith('config_fingerprint.json') for n in names) and any('checkpoints/' in n for n in names):
                            dest = os.path.join('exp8_extracted', os.path.splitext(fn)[0]); z.extractall(dest)
                            print(f'Extracted {p} -> {dest}'); return True
                except zipfile.BadZipFile:
                    pass
    return False

def show_inputs(max_entries=60):
    print('Nothing usable found. This is what is attached (first entries):')
    n = 0
    for root in INPUT_ROOTS:
        for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
            depth = dirpath[len(root):].count(os.sep)
            if depth > 3: continue
            print(f'  {dirpath}/  ({len(filenames)} files) ' + ', '.join(filenames[:5]))
            n += 1
            if n >= max_entries: return

if EXP8_DIR is None:
    hits = find_exp8()
    if not hits and unzip_exp8(): hits = find_exp8()
    if len(hits) > 1: hits = [h for h in hits if ('smoke' in h) == SMOKE_TEST] or hits
    if not hits:
        show_inputs()
        raise FileNotFoundError('Experiment 8 results not found. On Kaggle: right panel -> Add Input -> Your Work / Notebooks -> '
                                'select the COMMITTED Experiment 8 notebook (or upload its ZIP as a dataset), then Run All again. '
                                'Or set EXP8_DIR to the folder that contains config_fingerprint.json and checkpoints/.')
    EXP8_DIR = hits[0]
    print('Experiment 8 folders found:', hits)
FP8 = json.load(open(f'{EXP8_DIR}/config_fingerprint.json')); CFG = dict(FP8['config'])
CFG.update(top_free=[0.01, 0.05, 0.10, 0.25], random_draws=5)
OUT_DIR = 'exp8b_diagnostic' if not SMOKE_TEST else 'exp8b_smoke_test'
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device('cpu')                 # replay must run on CPU like the original run
def log(msg):
    line = f'[{datetime.datetime.now():%H:%M:%S}] {msg}'
    print(line, flush=True)
    with open(f'{OUT_DIR}/progress.log', 'a') as f: f.write(line + '\n')
ENV = dict(device=str(DEVICE), torch=torch.__version__, torch_geometric=torch_geometric.__version__,
           exp8_dir=EXP8_DIR, exp8_fingerprint=FP8['fingerprint'], started=f'{datetime.datetime.now():%Y-%m-%d %H:%M:%S}')
json.dump(ENV, open(f'{OUT_DIR}/environment.json', 'w'), indent=2)
log(f'Experiment 8 results: {EXP8_DIR} (fingerprint {FP8["fingerprint"]}) | datasets {CFG["datasets"]} | seeds {CFG["seeds"]}')

## 2. Pre-registration (descriptive diagnostic; saved before any result)

In [ ]:
PREREG = f'''# Experiment 8b: diagnostic on saved Experiment 8 checkpoints. Pre-registration
Saved: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}, before any result of this diagnostic. Experiment 8 fingerprint {FP8['fingerprint']}.
Experiment 8 and its pre-registered answers are not changed. All analyses here are descriptive (no pass/fail).

D1 replay check: RC-p0.00 is retrained from the saved Original with the Experiment 8 seed, bounds (biases frozen, row maximum
fixed), optimizer (Adam lr {CFG['ft_lr']}, weight decay {CFG['weight_decay']}) and {CFG['variant_steps']} steps, on CPU. Reported: max |W_replay - W_saved|.
If it exceeds 1e-4, D2 is labelled approximate.
D2 (weight-decay hypothesis for E3): per weight entry, over the replay, S_data = sum of |data gradient| and S_wd = sum of
|weight decay x w| (the two parts of the gradient Adam receives). For the top f = {CFG['top_free']} of entries by
|W_RC0 - W_O| / INT4 bin (the Experiment 8 TF ranking) and for the rest: share with S_wd > S_data ("weight-decay-gradient
dominance"), median S_data / S_wd, share moving toward zero (|W_RC0| < |W_O|), per layer and overall.
Sensitivity (actual optimizer updates): each Adam step's update of each entry (before projection) is attributed to weight decay
if it points toward zero and against (or without) the data gradient, to the data if it follows the data gradient and points
away from zero, and is left ambiguous when both agree; share of entries whose weight-decay-attributed movement exceeds the
data-attributed movement.
Hypothesis under test (proposed after Experiment 8): the top movers are mostly weight-decay dominated. It is reported
as consistent with the data only if both the gradient-dominance share and the update-attribution share are above 0.5 for the
top movers and clearly higher than for the rest; otherwise as not supported. Neither measure alone shows that weight decay
caused the movement: Adam's momentum and adaptive scaling act on the combined gradient.
D3 (reversion, no re-optimisation): from saved RC-p0.00 and FT models, revert to the Original (a) the top-f movers,
(b) all but the top-f movers, (c) the same number of random entries ({CFG['random_draws']} draws), and two code-based splits:
(d) Original-grid crossings: entries whose INT4 per-channel code differs when BOTH weights use the Original's scale (the
boundary-crossing mechanism analysis), and (e) the remaining in-cell entries; (f) recomputed-scale code changes: entries whose
code differs when each model uses its own scale (a deployment-quantization diagnostic), and (g) the rest. Biases keep the
model's own values. Measure forgetting achieved phi = 1 - |AUC - AUC(Retrain-1)| / |AUC(Original) - AUC(Retrain-1)| on the same
attack as Experiment 8. Paired over seeds, 95 % t-intervals. Reverting changes to their FP32 Original values is not the same as
quantizing: quantization maps weights to discrete values and does not restore them.
Limits: D3 removes changes without letting the network re-adapt, so it measures contribution in place, not necessity.
Pre-run clarifications after review: D2 is labelled weight-decay-gradient dominance and an optimizer-update sensitivity check
was added; D3 separates Original-grid crossings from recomputed-scale code changes, and no reversion is described as
equivalent to INT4 quantization. Objectives, replay check, seeds and controls are unchanged.
'''
path = f'{OUT_DIR}/exp8b_preregistration.md'
if os.path.exists(path): log('Pre-registration already saved earlier: keeping the original file unchanged.')
else: open(path, 'w').write(PREREG); log('Pre-registration saved.')
print(open(path).read())

## 3. Model (identical to Experiment 6)

In [ ]:
def norm_adj(edge_index, n):
    ei, w = gcn_norm(edge_index, None, n, add_self_loops=True)
    return torch.sparse_coo_tensor(torch.stack([ei[1], ei[0]]), w, (n, n)).coalesce().to(DEVICE)

class Layer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o, bias=False); self.bias = nn.Parameter(torch.zeros(o))
        nn.init.xavier_uniform_(self.lin.weight)
    def forward(self, x, A):
        xw = torch.sparse.mm(x, self.lin.weight.t()) if x.is_sparse else x @ self.lin.weight.t()
        return torch.sparse.mm(A, xw) + self.bias

class GCN(nn.Module):
    def __init__(self, i, h, o, p):
        super().__init__(); self.c1, self.c2, self.p = Layer(i, h), Layer(h, o), p
    def forward(self, xs, A):
        if self.training and self.p > 0:
            xs = torch.sparse_coo_tensor(xs.indices(), F.dropout(xs.values(), self.p, True), xs.shape)
        h = F.relu(self.c1(xs, A)); h = F.dropout(h, self.p, self.training)
        return self.c2(h, A)

# parameter order: c1.bias, c1.lin.weight (2-D), c2.bias, c2.lin.weight (2-D); weights are always selected by shape.
_d = Planetoid('data', 'Cora', transform=T.NormalizeFeatures())[0]
_m = GCN(_d.num_features, 128, 7, 0.5).to(DEVICE).eval()
assert [p.dim() for p in _m.parameters()] == [1, 2, 1, 2]
_r1, _r2 = GCNConv(_d.num_features, 128).to(DEVICE), GCNConv(128, 7).to(DEVICE)
with torch.no_grad():
    _m.c1.bias.add_(0.1)
    _r1.lin.weight.copy_(_m.c1.lin.weight); _r1.bias.copy_(_m.c1.bias); _r2.lin.weight.copy_(_m.c2.lin.weight); _r2.bias.copy_(_m.c2.bias)
    diff = (_m(_d.x.to_sparse().to(DEVICE), norm_adj(_d.edge_index, _d.num_nodes))
            - _r2(F.relu(_r1(_d.x.to(DEVICE), _d.edge_index.to(DEVICE))), _d.edge_index.to(DEVICE))).abs().max().item()
log(f'Equivalence check vs PyG GCNConv: max |difference| = {diff:.2e}')
assert diff < 1e-4, 'implementation does not match GCNConv'

## 4. Quantizer, matched noise, measurements (identical to the Experiment 6 quantization stage)

In [ ]:
def quantize(W, bits, scheme, scale_src=None):
    qmax = 2 ** (bits - 1) - 1; out, scales, clipped, total = [], [], 0, 0
    src = W if scale_src is None else scale_src
    for w, s0 in zip(W, src):
        if w.dim() != 2: out.append(w.clone()); scales.append(None); continue
        s = (s0.abs().amax(1, keepdim=True) if scheme == 'per_channel' else s0.abs().max()) / qmax
        s = torch.clamp(s, min=1e-12).expand_as(w)
        q = torch.round(w / s); clipped += int((q.abs() > qmax).sum()); total += q.numel()
        out.append(torch.clamp(q, -qmax, qmax) * s); scales.append(s)
    return out, scales, clipped / total

def matched_noise(W, Wq, scheme, gen):
    out = []
    for w, wq in zip(W, Wq):
        if w.dim() != 2: out.append(w.clone()); continue
        e = (wq - w).cpu()
        if scheme == 'per_channel':
            idx = torch.argsort(torch.rand(e.shape, generator=gen), dim=1); e = torch.gather(e, 1, idx)
        else:
            e = e.flatten()[torch.randperm(e.numel(), generator=gen)].view_as(e)
        out.append(w + e.to(w.device))
    return out

def flat2d(W): return torch.cat([w.flatten() for w in W if w.dim() == 2])
def cos(a, b): return float((a @ b) / (a.norm() * b.norm() + 1e-30))

_w = [torch.tensor([[0.70, -0.35, 0.10], [0.07, 0.02, -0.01]]), torch.zeros(2)]
assert torch.allclose(quantize(_w, 4, 'per_channel')[0][0], torch.tensor([[0.7, -0.4, 0.1], [0.07, 0.02, -0.01]]), atol=1e-6)
assert torch.allclose(quantize(_w, 4, 'per_tensor')[0][0], torch.tensor([[0.7, -0.4, 0.1], [0.1, 0.0, 0.0]]), atol=1e-6)
_q, _s, _c = quantize([_w[0] * 2, _w[1]], 4, 'per_channel', scale_src=_w)
assert abs(_c - 2 / 6) < 1e-9 and torch.allclose(_q[0][0], torch.tensor([0.7, -0.7, 0.2]), atol=1e-6)
_n = matched_noise(_w, quantize(_w, 4, 'per_channel')[0], 'per_channel', torch.Generator().manual_seed(0))
assert torch.allclose((quantize(_w, 4, 'per_channel')[0][0] - _w[0]).sort(1).values, (_n[0] - _w[0]).sort(1).values, atol=1e-7)
log('Quantizer and matched-noise self-checks passed.')

_DATA, _PROBE = {}, {}
def get_data(name):
    if name not in _DATA:
        d = Planetoid('data', name, transform=T.NormalizeFeatures())[0]
        _DATA[name] = dict(d=d, xs=d.x.to_sparse().to(DEVICE), y=d.y.to(DEVICE), y_np=d.y.numpy(),
                           A_full=norm_adj(d.edge_index, d.num_nodes), C=int(d.y.max()) + 1)
    return _DATA[name]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def make_split(ds, seed):
    G = get_data(ds); d = G['d']; N = d.num_nodes
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N); n_tr = int(CFG['train_frac'] * N)
    tr, held = np.sort(perm[:n_tr]), np.sort(perm[n_tr:])
    D = np.sort(rng.choice(tr, int(CFG['delete_frac'] * len(tr)), replace=False)); retained = np.setdiff1d(tr, D)
    hp = rng.permutation(held); H_cal, H_eval = hp[: len(hp) // 2], hp[len(hp) // 2:]
    R_cal = rng.choice(retained, size=min(len(retained), len(H_cal)), replace=False)
    return dict(train=tr, held=held, D=D, retained=retained, H_cal=H_cal, H_eval=H_eval, R_cal=R_cal)

def make_ctx(ds, sp):
    G = get_data(ds); N = G['d'].num_nodes
    keep = torch.ones(N, dtype=torch.bool); keep[torch.from_numpy(sp['D'])] = False
    ei_red, _ = subgraph(keep, G['d'].edge_index, num_nodes=N)
    if ds not in _PROBE: _PROBE[ds] = GCN(G['d'].num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
    return dict(ds=ds, G=G, sp=sp, A_red=norm_adj(ei_red, N), N=N)

def calibrate(cr, ch):
    cand = np.unique(np.r_[cr, ch]); sr, sh = np.sort(cr), np.sort(ch)
    tpr = 1 - np.searchsorted(sr, cand, 'left') / len(sr); tnr = np.searchsorted(sh, cand, 'left') / len(sh)
    return float(cand[int(np.argmax((tpr + tnr) / 2))])

@torch.no_grad()
def measure(ctx, W, thr_fp32=None):
    G, sp, probe = ctx['G'], ctx['sp'], _PROBE[ctx['ds']]
    for p, w in zip(probe.parameters(), W): p.data = w.clone()
    probe.eval(); y = G['y_np']; N = ctx['N']
    conf = F.softmax(probe(G['xs'], G['A_full']), 1).cpu().numpy()[np.arange(N), y]
    pred_red = probe(G['xs'], ctx['A_red']).argmax(1).cpu().numpy()
    D, He, ret = sp['D'], sp['H_eval'], sp['retained']
    auc = roc_auc_score(np.r_[np.ones(len(D)), np.zeros(len(He))], np.r_[conf[D], conf[He]])
    thr = calibrate(conf[sp['R_cal']], conf[sp['H_cal']]); t2 = thr if thr_fp32 is None else thr_fp32
    m = dict(auc=auc, adv=2 * abs(auc - 0.5), threshold=thr,
             tpr_minus_fpr_recal=(conf[D] >= thr).mean() - (conf[He] >= thr).mean(),
             tpr_minus_fpr_fp32_threshold=(conf[D] >= t2).mean() - (conf[He] >= t2).mean(),
             mean_conf_D=float(conf[D].mean()), mean_conf_Heval=float(conf[He].mean()),
             auc_retained_vs_Heval=roc_auc_score(np.r_[np.ones(len(ret)), np.zeros(len(He))], np.r_[conf[ret], conf[He]]),
             test_acc=float((pred_red[sp['held']] == y[sp['held']]).mean()), acc_deleted=float((pred_red[D] == y[D]).mean()))
    return m, conf

def node_recovery(sp, conf_new, conf_ft, conf_o):
    out = {}
    for name, idx in [('D', sp['D']), ('Heval', sp['H_eval'])]:
        t, dlt = conf_o[idx] - conf_ft[idx], conf_new[idx] - conf_ft[idx]; rms = float(np.sqrt(np.mean(t ** 2)))
        out.update({f'node_recovery_{name}': float(dlt @ t / (t @ t + 1e-30)), f'raw_shift_{name}': float(dlt.mean()),
                    f'raw_target_gap_{name}': float(t.mean()), f'target_rms_{name}': rms, f'unstable_{name}': bool(rms < CFG['unstable_rms'])})
    out['raw_shift_D_minus_Heval'] = out['raw_shift_D'] - out['raw_shift_Heval']
    return out

def ci95(x):
    x = np.asarray(x, float); n = len(x); m = x.mean()
    if n < 2: return m, np.nan, np.nan
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n); return m, m - h, m + h

def retrain_band(auc_by_seed):
    cols = [c for c in auc_by_seed.columns if c.startswith('Retrain-')]
    sq = [(auc_by_seed[a] - auc_by_seed[b]) ** 2 for a, b in itertools.combinations(cols, 2)]
    return float(2 * np.sqrt(pd.concat(sq).mean()))

## 5. Helpers: checkpoints, bins, forgetting achieved

In [ ]:
QMAX = 2 ** (CFG['cell_bits'] - 1) - 1
def load_ck(ds, seed):
    ck = torch.load(f'{EXP8_DIR}/checkpoints/{ds}_seed{seed}.pt', weights_only=False)
    assert ck.get('fingerprint') == FP8['fingerprint'], 'checkpoint fingerprint does not match the Experiment 8 folder'
    return {k: [w.to(DEVICE) for w in v] for k, v in ck['weights'].items()}, ck['split']

def bins(w0): return (w0.abs().amax(1, keepdim=True) / QMAX).expand_as(w0)
def codes(w, s): return torch.round(w / s)

def phi_of(ctx, W, a_o, a_r):
    a = measure(ctx, W)[0]['auc']; return 1 - abs(a - a_r) / abs(a_o - a_r), a

def revert(W, W0, masks):
    # masks: per 2-D tensor, True = revert this entry to the Original; biases untouched
    out, j = [], 0
    for w, w0 in zip(W, W0):
        if w.dim() != 2: out.append(w.clone()); continue
        out.append(torch.where(masks[j], w0, w)); j += 1
    return out

def topk_mask(rel, f):
    k = max(1, int(round(f * rel.numel()))); thr = torch.topk(rel.flatten(), k).values.min()
    return rel >= thr
log('Helpers ready.')

## 6. D1-D2: replay RC-p0.00 with gradient bookkeeping

In [ ]:
def bounds_rc0(W0):
    B = []
    for w0 in W0:
        if w0.dim() != 2: B.append((w0.clone(), w0.clone())); continue
        inner = w0.abs().amax(1, keepdim=True) * (1 - 1e-6)
        lo, hi = (-inner).expand_as(w0).clone(), inner.expand_as(w0).clone()
        am = torch.zeros_like(w0, dtype=torch.bool); am[torch.arange(w0.shape[0]), w0.abs().argmax(1)] = True
        B.append((torch.where(am, w0, lo), torch.where(am, w0, hi)))
    return B

def replay(ds, seed, W, sp):
    G = get_data(ds); d, xs, Y = G['d'], G['xs'], G['y']; ctx = make_ctx(ds, sp); N = d.num_nodes
    m_red = torch.zeros(N, dtype=torch.bool); m_red[torch.from_numpy(sp['retained'])] = True
    set_seed(5000 + seed + int(CFG['ft_lr'] * 1e4))
    m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout'])
    for p, w in zip(m.parameters(), W['Original']): p.data = w.clone()
    opt = torch.optim.Adam(m.parameters(), lr=CFG['ft_lr'], weight_decay=CFG['weight_decay']); B = bounds_rc0(W['Original'])
    S_data = [torch.zeros_like(p) for p in m.parameters()]; S_wd = [torch.zeros_like(p) for p in m.parameters()]
    U_wd = [torch.zeros_like(p) for p in m.parameters()]; U_data = [torch.zeros_like(p) for p in m.parameters()]
    for _ in range(CFG['variant_steps']):
        m.train(); opt.zero_grad(); F.cross_entropy(m(xs, ctx['A_red'])[m_red], Y[m_red]).backward()
        with torch.no_grad():
            before = [p.data.clone() for p in m.parameters()]; gd = [p.grad.clone() for p in m.parameters()]
            for i, p in enumerate(m.parameters()):          # p.grad = data gradient only (Adam adds weight decay internally)
                S_data[i] += p.grad.abs(); S_wd[i] += (CFG['weight_decay'] * p.data).abs()
        opt.step()
        with torch.no_grad():                               # attribute the actual Adam update (before projection)
            for i, p in enumerate(m.parameters()):
                du = p.data - before[i]; toward0 = torch.sign(du) == -torch.sign(before[i]); follows = torch.sign(du) == -torch.sign(gd[i])
                U_wd[i] += torch.where(toward0 & ~follows, du.abs(), torch.zeros_like(du))
                U_data[i] += torch.where(follows & ~toward0, du.abs(), torch.zeros_like(du))
        with torch.no_grad():
            for p, (lo, hi) in zip(m.parameters(), B): p.data = torch.minimum(torch.maximum(p.data, lo), hi)
    Wr = [p.detach().clone() for p in m.parameters()]
    diff = max(float((a - b).abs().max()) for a, b in zip(Wr, W['RC-p0.00']))
    return Wr, S_data, S_wd, U_wd, U_data, diff

D2_PATH = f'{OUT_DIR}/d2_weight_decay.csv'; rows, rep = [], []
if os.path.exists(D2_PATH): log('D2 already computed: loading.'); D2 = pd.read_csv(D2_PATH); REP = pd.read_csv(f'{OUT_DIR}/d1_replay.csv')
else:
    for ds in CFG['datasets']:
        for seed in CFG['seeds']:
            t0 = time.time(); W, sp = load_ck(ds, seed); Wr, Sd, Sw, Uw, Ud, diff = replay(ds, seed, W, sp)
            rep.append(dict(dataset=ds, seed=seed, max_abs_diff_vs_saved=diff, exact=bool(diff <= 1e-4)))
            j = 0
            for w0, w1, sd, sw, uw, ud in zip(W['Original'], W['RC-p0.00'], Sd, Sw, Uw, Ud):
                if w0.dim() != 2: continue
                j += 1; rel = (w1 - w0).abs() / bins(w0); wd_dom = sw > sd; toward0 = w1.abs() < w0.abs(); ratio = sd / (sw + 1e-30)
                upd_wd = uw > ud
                for f in CFG['top_free']:
                    top = topk_mask(rel, f)
                    for grp, msk in [('top', top), ('rest', ~top)]:
                        rows.append(dict(dataset=ds, seed=seed, layer=j, f=f, group=grp, n=int(msk.sum()),
                                         share_wd_gradient_dominant=float(wd_dom[msk].float().mean()),
                                         share_wd_update_attributed=float(upd_wd[msk].float().mean()), median_data_over_wd=float(ratio[msk].median()),
                                         share_toward_zero=float(toward0[msk].float().mean()), median_move_bins=float(rel[msk].median())))
            log(f'{ds} seed {seed}: replay done in {time.time()-t0:.0f}s, max |replay - saved| = {diff:.2e}')
    D2 = pd.DataFrame(rows); D2.to_csv(D2_PATH, index=False); REP = pd.DataFrame(rep); REP.to_csv(f'{OUT_DIR}/d1_replay.csv', index=False)
print('D1 replay check:'); display(REP)
w = lambda g: np.average(g, weights=D2.loc[g.index, 'n'])
summ = D2.groupby(['dataset', 'f', 'group']).agg(share_wd_gradient_dominant=('share_wd_gradient_dominant', w),
                                                 share_wd_update_attributed=('share_wd_update_attributed', w), share_toward_zero=('share_toward_zero', w),
                                                 median_data_over_wd=('median_data_over_wd', 'median'), median_move_bins=('median_move_bins', 'median')).reset_index()
summ.to_csv(f'{OUT_DIR}/d2_summary.csv', index=False); print('D2 (weighted over layers and seeds):'); display(summ.round(3))

## 7. D3: revert selected weight changes, measure forgetting achieved

In [ ]:
D3_PATH = f'{OUT_DIR}/d3_reversion.csv'
if os.path.exists(D3_PATH): log('D3 already computed: loading.'); D3 = pd.read_csv(D3_PATH)
else:
    rows = []
    for ds in CFG['datasets']:
        for seed in CFG['seeds']:
            W, sp = load_ck(ds, seed); ctx = make_ctx(ds, sp)
            a_o, a_r = measure(ctx, W['Original'])[0]['auc'], measure(ctx, W['Retrain-1'])[0]['auc']
            W0 = W['Original']; W02 = [w for w in W0 if w.dim() == 2]
            for model in ['RC-p0.00', f"FT dose={CFG['ft_dose']}"]:
                Wm = W[model]; Wm2 = [w for w in Wm if w.dim() == 2]
                rel = [(a - b).abs() / bins(b) for a, b in zip(Wm2, W02)]
                cross = [codes(a, bins(b)) != codes(b, bins(b)) for a, b in zip(Wm2, W02)]     # Original grid for both weights
                changed = [codes(a, bins(a)) != codes(b, bins(b)) for a, b in zip(Wm2, W02)]   # each model's own scale
                base_phi, _ = phi_of(ctx, Wm, a_o, a_r)
                def add(cond, f, masks, draw=-1):
                    ph, a = phi_of(ctx, revert(Wm, W0, masks), a_o, a_r)
                    rows.append(dict(dataset=ds, seed=seed, model=model, condition=cond, f=f, draw=draw, phi=ph, auc=a, phi_unreverted=base_phi,
                                     share_reverted=float(sum(int(m.sum()) for m in masks) / sum(m.numel() for m in masks))))
                add('none', 0, [torch.zeros_like(m) for m in changed])
                add('revert Original-grid crossings', 0, cross)
                add('revert Original-grid in-cell changes', 0, [~m for m in cross])
                add('revert recomputed-scale code changes', 0, changed)
                add('revert recomputed-scale unchanged codes', 0, [~m for m in changed])
                for f in CFG['top_free']:
                    tops = [topk_mask(r, f) for r in rel]
                    add('revert top-f movers', f, tops); add('revert all but top-f movers', f, [~t for t in tops])
                    for dr in range(CFG['random_draws']):
                        g = torch.Generator().manual_seed(int(1e6 * f) + 100 * seed + dr)
                        add('revert random f entries', f, [torch.rand(t.shape, generator=g) < f for t in tops], dr)
            log(f'{ds} seed {seed}: reversion tests done')
    D3 = pd.DataFrame(rows); D3.to_csv(D3_PATH, index=False)
def ci(x):
    x = np.asarray(x, float); m = x.mean(); h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
    return pd.Series(dict(phi=m, ci_lo=m - h, ci_hi=m + h))
per_seed = D3.groupby(['dataset', 'model', 'condition', 'f', 'seed']).agg(phi=('phi', 'mean'), share_reverted=('share_reverted', 'mean')).reset_index()
S3 = per_seed.groupby(['dataset', 'model', 'condition', 'f']).phi.apply(ci).unstack().reset_index()
S3 = S3.merge(per_seed.groupby(['dataset', 'model', 'condition', 'f']).share_reverted.mean().reset_index(), on=['dataset', 'model', 'condition', 'f'])
S3.to_csv(f'{OUT_DIR}/d3_summary.csv', index=False)
pd.set_option('display.width', 220)
for ds in CFG['datasets']:
    for model in S3.model.unique():
        print(f'\n=== {ds}, {model}: forgetting achieved phi after reverting (1 = matches Retrain, 0 = back to Original) ===')
        display(S3[(S3.dataset == ds) & (S3.model == model)].drop(columns=['dataset', 'model']).round(3))

## 8. Plot and export

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(CFG['datasets']), figsize=(7 * len(CFG['datasets']), 4.5), sharey=True); axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CFG['datasets']):
    s = S3[(S3.dataset == ds) & (S3.model == 'RC-p0.00')]
    for cond, col in [('revert top-f movers', 'C3'), ('revert random f entries', 'C7'), ('revert all but top-f movers', 'C0')]:
        g = s[s.condition == cond].sort_values('f')
        ax.errorbar(g.f * 100, g.phi, yerr=[g.phi - g.ci_lo, g.ci_hi - g.phi], marker='o', capsize=3, color=col, label=cond)
    for cond, ls in [('none', '-'), ('revert Original-grid in-cell changes', '--'), ('revert Original-grid crossings', ':')]:
        ax.axhline(float(s[s.condition == cond].phi.iloc[0]), ls=ls, c='k', lw=1, label=cond)
    ax.set_xscale('log'); ax.set_xlabel('share of entries selected (%)'); ax.set_title(f'{ds}: reverting parts of RC-p0.00')
axes[0].set_ylabel('forgetting achieved phi'); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_reversion.png', dpi=160); plt.show()
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR); log(f'ZIP created: {OUT_DIR}.zip')
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    from IPython.display import FileLink, display; display(FileLink(f'{OUT_DIR}.zip'))